# Python User-Defined Function (UDF) to Detect Sentiment
This notebook creates and uses a Python UDF that:
- reads unstructured data from a Snowflake stage (directory table)
- uses a pre-built machine learning model to label the data 


The function also reads a configuration file and writes it to **/tmp** in the local file system of the Snowflake virtual warehouse runtime for code use.

The resulting sentiment scores are then integrated with structured and semi-structured data for straightforward SQL reporting.

---
### The unstructured data for labeling
Along with structured and semi-structured data for vending machines, products, and transactions, this example uses a stage that contains sample customer reviews of vending machine purchases. 

The files in the stage **@reviewdata.reviews** are of the form:

File name: `<vending_transaction>_<transaction_item>.txt`<br>
File contents: `<customer review>`

Example:

File name: `b15a2d41ccccef7ce3f64d844e6fe7cd1f5495ec_1.txt`<br>
File contents: `crisp and fresh`

**Note on file name**: A single sales transaction can contain multiple items purchased and can receive multiple reviews. A customer can review any particular item purchased and/or can review the purchase transaction in general. In this data set, the file names act as "intelligent keys", including the *vending_transaction* and the *transaction_item* for the particular item purchased. *transaction_item* = 0 for a general review of a purchase. These parts of the file name can be extracted to join directly to the **VENDING_TXN** DataFrame created below, on columns **(TXN, TXN_ITEM)**. 

**Note on file contents**: Each review contains natural language contained in a simple text file. These short strings are presented here for simplicity, but the same programming techniques can be used to process *any other type* of unstructured data.

### The function
The notebook creates function **udfs.sentiment_score(URL)**, which takes the natural language of a customer review and estimates the sentiment of the review, using the [NLTK VADER sentiment analyzer](https://www.nltk.org/_modules/nltk/sentiment/vader.html) included in the Anaconda distribution provided for Snowflake Python.

The function takes a URL for an unstructured data record and returns an object reporting the sentiment inferred. Using best practices for Snowflake code security, the function must be called with a *scoped URL* as its argument.

Example invocation:

`sentiment_score(`<br>
`   build_scoped_file_url(`<br>
`      @util_db.data.reviews, 'b15a2d41ccccef7ce3f64d844e6fe7cd1f5495ec_1.txt'`<br>
`   )`<br>
`)`

Return:

`{"compound": 0.3182, "neg": 0, "neu": 0.465, "pos": 0.535, "review": "crisp and fresh", "sentiment": "POSITIVE"}`

In addition to the numeric scores returned by the VADER sentiment analyzer, the function includes the review text and a final sentiment label (POSITIVE, NEUTRAL, or NEGATIVE, based on a simple case statement on the "compound" numeric score.)

---
### Steps below
1. Connect to Snowflake
1. Create Python UDF to label unstructured data
1. Test the function
1. Batch Scoring 
1. Reporting
1. Lab Cleanup

In [1]:
import snowflake.snowpark
from snowflake.snowpark.functions import *
from snowflake.snowpark.session import Session

import pandas as pd

# config_dir = '/home/jovyan/.ssh'
# configfile = config_dir + '/sf_config'

### 1. Connect to Snowflake

In [2]:
session = Session.builder.configs({
      "account":   "ES10286-ML_ENTERPRISE",
      "user":      "RKIRK",
      "password":  "8d!upvFs2#BDDB5JQ*7",
      "role":      "RK_SANDPIT_SYSADMIN",
      "warehouse": "DAFT_WH",
      "database":  "RK_SANDPIT",
      "schema":    "DAFT_SCHEMA"
  }).create()

---
### 2. Create Python UDF to label unstructured data

**Note on the code:** The sentiment analyzer is created with declaration `SentimentIntensityAnalyzer(lexicon_file)`. The Python class, as written, loads the lexicon of sentiment words *from the local disk file* given in the invocation. To meet this requirement, the UDF must place the lexicon file in the local disk environment of any Snowflake virtual warehouse that runs the sentiment analyzer.

- A standard lexicon file of sentiment words can be obtained with the command `nltk.download('vader_lexicon')`.
- This file has been acquired and placed in a well-known stage in the **UTIL** schema.
- To enable local files in the runtime environment, Snowflake allows a UDF or stored procedure to write to the local **/tmp** folder only.
- The UDF checks for the existence of the file and, if absent, reads from the stage and writes it to the local **/tmp** folder.

See the following lines within the code below:

`LEXICON_STAGE_FILE_URI = '@util.lexicon/vader_lexicon.txt'`<br>
`LEXICON_LOCAL_TMP_FILE = '/tmp/vader_lexicon.txt'`

These program constants identify the stage and file containing the lexicon and the local **/tmp** location where the file will be placed for loading by the sentiment analyzer declaration.

#### -> Define the Python function for sentiment analysis

In [7]:
# Imports
from snowflake.snowpark.files import SnowflakeFile
import io
import os
from os import path

def retrieve_sentiment_scores(review_url):
    
    # The nltk package is available in the Python sandbox of a Snowflake
    # virtual warehouse. 
    # By placing the following import within the handler of the UDF, we don't
    # need nltk to be installed in the development environment.
    
    from nltk.sentiment.vader import SentimentIntensityAnalyzer
    
    LEXICON_STAGE_FILE_URI = '@util.lexicon/vader_lexicon.txt' # Relative to current database
    LEXICON_LOCAL_TMP_FILE = '/tmp/vader_lexicon.txt'
    
    # Check to see if the lexicon file exists locally. If not, then
    # copy staged file LEXICON_STAGE_FILE_URI to local LEXICON_LOCAL_TMP_FILE.
    if not path.isfile(LEXICON_LOCAL_TMP_FILE):
        output = os.open(LEXICON_LOCAL_TMP_FILE, os.O_RDWR|os.O_CREAT|os.O_TRUNC)
        with SnowflakeFile.open(LEXICON_STAGE_FILE_URI, 'rb', require_scoped_url = False) as input:
            wrapped_text_file = io.TextIOWrapper(input, encoding='utf-8')
            for line in wrapped_text_file:
                os.write(output, bytes(line, 'utf-8'))
        os.close(output)

    # Declare analyzer using locally installed lexicon: RK Fix here required for latest NLTK package
    # analyzer = SentimentIntensityAnalyzer(LEXICON_LOCAL_TMP_FILE)
    analyzer = SentimentIntensityAnalyzer('file:' + LEXICON_LOCAL_TMP_FILE)

    # Get review from scoped URL
    review_file = SnowflakeFile.open(review_url, 'r')
    review = review_file.readline().strip()

    # Score the review
    scores = analyzer.polarity_scores(review)

    # Resolve sentiment numeric score to a general sentiment
    if scores['compound'] >= 0.05 :
        scores['sentiment'] = 'POSITIVE'
    elif scores['compound'] <= -0.05 :
        scores['sentiment'] = 'NEGATIVE'
    else :
        scores['sentiment'] = 'NEUTRAL'

    # Include the review itself
    scores["review"] = review

    return scores

#### -> Register as a UDF

In [8]:
# Create a stage to hold the permanent UDF.
session.sql("create schema if not exists udfs").collect()
session.sql("create stage if not exists udfs.udfcode").collect()

# Imports
from snowflake.snowpark.types import StringType, VariantType

# Register the UDF
sentiment_udf = (session
    .udf                                          # An instance of UDFRegistration
    .register(                         
         func = retrieve_sentiment_scores         # The Python function powering the UDF                                       
        ,name = "udfs.sentiment_score"  # The name of the UDF    
        ,return_type = VariantType()
        ,input_types = [StringType()]
        ,is_permanent = True
        ,stage_location = "@udfs.udfcode"  # Where to store the pickled bytecode (if large enough)
        ,packages = ["snowflake-snowpark-python", "nltk"]
        ,replace = True
        ,session = session
    )
)

Package 'nltk' is not installed in the local environment. Your UDF might not work when the package is installed on the server but not on your local environment.


My notes: The above code creates schema and function in the schema: RK_SANDPIT.UDFS.SENTIMENT_SCORE

---
### 3. Test the function

In [9]:
session.sql("""
select relative_path, 
       udfs.sentiment_score(build_scoped_file_url(@reviewdata.reviews, relative_path)) sentiment
from directory(@reviewdata.reviews)
limit 1""").show(1, 100)

-----------------------------------------------------------------------------------------------------------------------------------------
|"RELATIVE_PATH"                                 |"SENTIMENT"                                                                           |
-----------------------------------------------------------------------------------------------------------------------------------------
|0203151e7b76fe62238b1d70d3bbe29945a157ce_1.txt  |{                                                                                     |
|                                                |  "compound": 3.400000000000000e-01,                                                  |
|                                                |  "neg": 1.440000000000000e-01,                                                       |
|                                                |  "neu": 6.250000000000000e-01,                                                       |
|                                 

My notes: Can also call the function by using the SQL directly in snowsight:
```sql
select relative_path,udfs.sentiment_score(build_scoped_file_url(@reviewdata.reviews, relative_path)) sentiment
from directory(@reviewdata.reviews)
limit 1
;
```

---
### 4. Batch Scoring

#### -> Collect all reviews and their sentiment scores in a table.

In [10]:
# Create DataFrame of contents
sentimentsDF = session.sql("""
select 
    substr(relative_path, 1, position('_' in relative_path)-1)        txn
    , replace(split_part(relative_path, '_', 2), '.txt', '')::int     txn_item
    , udfs.sentiment_score(
        build_scoped_file_url(@reviewdata.reviews, relative_path))  sentiment
from directory(@reviewdata.reviews)
limit 10000""")

In [11]:
%%time

# Save to a table
sentimentsDF. \
 write. \
 mode('overwrite'). \
 save_as_table('public.scored_sentiments')

CPU times: user 41.3 ms, sys: 4.9 ms, total: 46.2 ms
Wall time: 40.1 s


#### -> Confirm result

In [12]:
session.sql('list @reviewdata.reviews').count()   # Count of reviews in the stage

146

In [13]:
session.table('public.scored_sentiments').count()    # Count of reviews scored in the flattened table

146

In [16]:
session.sql('describe table public.scored_sentiments').show()   # Structure of the table

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"name"     |"type"              |"kind"  |"null?"  |"default"  |"primary key"  |"unique key"  |"check"  |"expression"  |"comment"  |"policy name"  |"privacy domain"  |"write default"  |
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|TXN        |VARCHAR(134217728)  |COLUMN  |Y        |NULL       |N              |N             |NULL     |NULL          |NULL       |NULL           |NULL              |NULL             |
|TXN_ITEM   |NUMBER(38,0)        |COLUMN  |Y        |NULL       |N              |N             |NULL     |NULL          |NULL       |NULL           |NULL              |NULL             |
|SENTIMENT  |VARIANT             |COLUMN  |Y        |NULL       |

In [17]:
session.sql("""select * from public.scored_sentiments limit 2""").show()   # Convenience sample

--------------------------------------------------------------------------------------------------------------
|"TXN"                                     |"TXN_ITEM"  |"SENTIMENT"                                         |
--------------------------------------------------------------------------------------------------------------
|0203151e7b76fe62238b1d70d3bbe29945a157ce  |1           |{                                                   |
|                                          |            |  "compound": 3.400000000000000e-01,                |
|                                          |            |  "neg": 1.440000000000000e-01,                     |
|                                          |            |  "neu": 6.250000000000000e-01,                     |
|                                          |            |  "pos": 2.310000000000000e-01,                     |
|                                          |            |  "review": "Fizzy Frenzy has a lot of fizz and...  |
|

---
### 5. Reporting

#### -> Initial setup

Note the table **PUBLIC.VENDING_TXN** with content that joins directly to the sample customer reviews just processed. 

We'll create a DataFrame on that data that flattens the `VARIANT` column into a set of structured columns. The resulting DataFrame and our **SCORED_SENTIMENTS** table can be easily joined on columns **(TXN, TXN_ITEM)**.

In [18]:
from snowflake.snowpark.types import *
transactions = (
    session.table('PUBLIC.VENDING_TXN')
    .withColumn('txn', col('data')['TXN'].cast(StringType()))
    .withColumn('txn_time', col('data')['TXN_TIME'].cast(TimestampType()))
    .withColumn('txn_item', col('data')['TXN_ITEM'].cast(IntegerType()))
    .withColumn('vm_id', col('data')['VM_ID'].cast(IntegerType()))
    .withColumn('status', col('data')['STATUS'].cast(StringType()))
    .withColumn('sale_total', col('data')['SALE_TOTAL'].cast(DecimalType(38,2)))
    .withColumn('quant', col('data')['QUANT'].cast(IntegerType()))
    .withColumn('prd_id', col('data')['PRD_ID'].cast(IntegerType()))
    .withColumn('payment_id', col('data')['PAYMENT_ID'].cast(StringType()))
    .withColumn('price', col('data')['PRICE'].cast(DecimalType(38,2)))
    .withColumn('p_category', col('data')['P_CATEGORY'].cast(StringType()))
    .withColumn('p_name', col('data')['P_NAME'].cast(StringType()))
    .drop('data')
)

Create DataFrames on additional datasets of interest.

In [19]:
reviews = session.table('public.scored_sentiments')          # The scored customer reviews
products = session.table('public.product')                   # Product details
vending_machines = session.table('public.vending_machine')   # Vending machine details

#### -> Review DataFrames for analysis

In [20]:
reviews.show(1)

--------------------------------------------------------------------------------------------------------------
|"TXN"                                     |"TXN_ITEM"  |"SENTIMENT"                                         |
--------------------------------------------------------------------------------------------------------------
|0203151e7b76fe62238b1d70d3bbe29945a157ce  |1           |{                                                   |
|                                          |            |  "compound": 3.400000000000000e-01,                |
|                                          |            |  "neg": 1.440000000000000e-01,                     |
|                                          |            |  "neu": 6.250000000000000e-01,                     |
|                                          |            |  "pos": 2.310000000000000e-01,                     |
|                                          |            |  "review": "Fizzy Frenzy has a lot of fizz and...  |
|

In [21]:
transactions.show(1)

---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"TXN"                                     |"TXN_TIME"           |"TXN_ITEM"  |"VM_ID"  |"STATUS"   |"SALE_TOTAL"  |"QUANT"  |"PRD_ID"  |"PAYMENT_ID"      |"PRICE"  |"P_CATEGORY"  |"P_NAME"                 |
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|6b99b1680bde367fd0e4334073c55eef64b7bc51  |2022-04-29 06:42:27  |1           |1        |processed  |2.00          |1        |2         |4365017719973021  |2.00     |FOOD          |Nutty Nibbles - Variety  |
------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [22]:
products.show(1)

----------------------------------------------------------------------------------------------------------------------------
|"PRD_ID"  |"NAME"          |"SELL_PRICE"  |"DESC"                                              |"CATEGORY"  |"BUY_PRICE"  |
----------------------------------------------------------------------------------------------------------------------------
|1         |Choco Loco Bar  |1.75          |The "Choco Loco Bar" is a delicious chocolate b...  |FOOD        |1.22         |
----------------------------------------------------------------------------------------------------------------------------



In [23]:
vending_machines.show(1)

-------------------------------------------------------------------
|"VM_ID"  |"LOC"                |"CAP"  |"STATUS"  |"BUILD_DATE"  |
-------------------------------------------------------------------
|1        |032 Patterson Drive  |300    |ACTIVE    |2018-05-04    |
-------------------------------------------------------------------



#### -> Questions and Insights

**Q: What are the reviews for Nutty Nibbles?**

In [24]:
answer = (
    reviews 
    .join(transactions, ['txn', 'txn_item']) 
    .join(products, 'prd_id') 
    .filter(col('name').like('Nutty Nibbles%')) 
    .withColumn('review', col('sentiment')['review'].cast(StringType())) 
    .withColumn('sentiment', col('sentiment')['sentiment'].cast(StringType())) 
    .select('review', 'sentiment')
)
rows = answer.count()
rows

10

In [26]:
answer.show(rows)

--------------------------------------------------------------------
|"REVIEW"                                            |"SENTIMENT"  |
--------------------------------------------------------------------
|I love Nutty Nibbles! Perfect balance of sweet ...  |POSITIVE     |
|Nutty Nibbles were hard and sour.                   |NEGATIVE     |
|Nutty Nibbles are bland and boring.                 |NEGATIVE     |
|I got a Nutty Nibbles from the vending machine ...  |NEGATIVE     |
|These nibbles are organic and vegan-friendly.       |NEUTRAL      |
|I got a Nutty Nibbles from the vending machine ...  |NEGATIVE     |
|I got some Nutty Nibbles and they were varied a...  |NEUTRAL      |
|I got some Nutty Nibbles. Hard\, dry and almond...  |NEGATIVE     |
|Nutty Nibbles is fine\, but Im allergic to some...  |NEGATIVE     |
|That bag of Nutty Nibbles was awful. Its sour a...  |NEGATIVE     |
--------------------------------------------------------------------



In [27]:
answer.show(rows,  200)   # Longer review text

----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"REVIEW"                                                                                                                                                                                        |"SENTIMENT"  |
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|I love Nutty Nibbles! Perfect balance of sweet and salty\, crunchy and chewy.                                                                                                                   |POSITIVE     |
|Nutty Nibbles were hard and sour.                                                                                                                                  

**Q: What are the general transaction reviews?**

In [28]:
answer = (
    reviews 
    .filter(col('txn_item')==0) 
    .withColumn('review', col('sentiment')['review'].cast(StringType())) 
    .withColumn('sentiment', col('sentiment')['sentiment'].cast(StringType())) 
    .select('review', 'sentiment')
)

rows = answer.count()
rows

7

In [29]:
answer.show(rows,200)

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"REVIEW"                                                                                                                                                                                    |"SENTIMENT"  |
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|Your vending machines are polite and respectful. They have friendly messages or sounds. I like them and thank them.                                                                         |POSITIVE     |
|Your vending machines are too expensive and overpriced. They charge more than the market value or the store price.                                                                 

**Q: Give net reviews on all products (Sum for each review: NEGATIVE = -1, NEUTRAL = 0, POSITIVE = 1)**

Analysis using DataFrame programming:

In [30]:
answer = (
    reviews 
    .join(transactions, ['txn', 'txn_item']) 
    .join(products, 'prd_id') 
    .withColumn('sentiment', col('sentiment')['sentiment'].cast(StringType())) 
    .withColumn('review_score', 
                when(col('sentiment') == 'NEGATIVE', lit(-1)) 
                .when(col('sentiment') == 'NEUTRAL', lit(0)) 
                .when(col('sentiment') == 'POSITIVE', lit(1)) 
                .otherwise(lit(0))) 
    .groupBy('name') 
    .agg(
        [count('*').alias('num_reviews') 
        ,sum('review_score').alias('net_sentiment')
        ]) 
    .withColumn('annotated_name', concat(col('name'), lit(' ('), col('num_reviews'), lit(')'))) 
    .select('annotated_name', 'net_sentiment') 
    .orderBy(col('net_sentiment').desc())
)

rows = answer.count()
rows

20

In [31]:
answer.show(rows, 60)

-------------------------------------------------------------------------
|"ANNOTATED_NAME"                                     |"NET_SENTIMENT"  |
-------------------------------------------------------------------------
|Tickle Tonic - Orange (10)                           |7                |
|Sparkling Shenanigans - Regular (7)                  |4                |
|Effervescent Elixir (6)                              |4                |
|Fizzy Frenzy Original (7)                            |3                |
|Placid Punch - Ginger Lemon (9)                      |2                |
|Bubbly Brew - Zero Sugar (7)                         |2                |
|Kettle Cooked Kicks - Salsa (10)                     |1                |
|Restful Raspberry Tea (4)                            |0                |
|Aqua LaLa Water (9)                                  |0                |
|Oatmeal Outburst Cookies (12)                        |0                |
|Bob Robert's - Fudgy Fun -  Chocolate

Equivalent SQL:

In [32]:
# Save DataFrame as a temporary view to support the following SQL statement
transactions.createOrReplaceView('public.vending_txn_vw')  

answer = session.sql(
    """
    with cte as (
        select p.name, 
            count(*) num_reviews,
            p.name || ' (' || num_reviews || ')' annotated_name,
            sum(case s.sentiment:sentiment::string
                    when 'POSITIVE' then 1
                    when 'NEUTRAL' then 0
                    when 'NEGATIVE' then -1
                end) net_sentiment
        from public.vending_txn_vw v 
            join public.product p using (prd_id)
            join public.scored_sentiments s using (txn, txn_item)
        group by 1
    )
    select annotated_name, net_sentiment
    from cte
    order by net_sentiment desc
    """)

rows = answer.count()
rows

20

In [33]:
answer.show(rows, 60)

-------------------------------------------------------------------------
|"ANNOTATED_NAME"                                     |"NET_SENTIMENT"  |
-------------------------------------------------------------------------
|Tickle Tonic - Orange (10)                           |7                |
|Effervescent Elixir (6)                              |4                |
|Sparkling Shenanigans - Regular (7)                  |4                |
|Fizzy Frenzy Original (7)                            |3                |
|Placid Punch - Ginger Lemon (9)                      |2                |
|Bubbly Brew - Zero Sugar (7)                         |2                |
|Kettle Cooked Kicks - Salsa (10)                     |1                |
|Restful Raspberry Tea (4)                            |0                |
|Oatmeal Outburst Cookies (12)                        |0                |
|Aqua LaLa Water (9)                                  |0                |
|Choco Loco Bar (1)                   

For convenience, you can save the answer DataFrame as a view for later requerying.

In [34]:
answer.createOrReplaceView('public.product_sentiment_vw')

[Row(status='View PRODUCT_SENTIMENT_VW successfully created.')]

Now the query above--or the simpler query `SELECT * FROM product_sentiment_vw`--can be executed in Snowsight, which can then be easily used to create a chart like this:

<img src="images/Product_Net_Reviews.png" alt="DFOperationsTransform" style="width:60%;display:block;margin-left:5%;" />


**Q: Show reviews of our vending machines.**

If you choose, define a DataFrame (and save as a view) in order to produce a report on net sentiment on vending machines, like the one below:

<img src="images/Reviews_by_Vending_Machine.png" alt="DFOperationsTransform" style="width:60%;display:block;margin-left:5%;" />


A solution has been provided in the file 03_exercise_solution.ipynb in case you want compare to your own solution.

## Summary

This notebook makes the following points:
- You can easily integrate unstructured data with other data in Snowflake by placing files in a stage, usually with a directory table on the stage.
- From an RDBMS perspective, each file in the stage is a record or row in the directory table; the file relative path is the key to the row.
- You can enable processing of the unstructured data with UDFs or procedures that open files using the `Snowflake.open()` method.